# Advanced `yield from` Tutorial
## Step-by-step problems with complete solutions

This notebook is a second, independent tutorial on the same topic:

- generators;
- coroutine-style `.send()`;
- delegation with `yield from`;
- subgenerator return values;
- exception forwarding;
- cleanup propagation;
- recursive lazy traversal;
- generator inspection.

The emphasis here is not only on final code. Each problem is broken into small
reasoning steps, with many explanatory cells between code cells.

---

## How to use this notebook

For every problem:

1. Read the scenario.
2. Predict the protocol before running code.
3. Complete or mentally solve each small step.
4. Compare your reasoning with the provided solution.
5. Run the assertions.
6. Change one assumption and add another test.

The examples use only the Python standard library.

## Learning map

We will progress through four layers:

### Layer 1 — The delegation protocol

We begin by comparing manual forwarding with `yield from`.

### Layer 2 — Bidirectional coroutine communication

We then send values inward, receive values outward, and capture returned summaries.

### Layer 3 — Exceptions and lifecycle

We study `.throw()`, `.close()`, `GeneratorExit`, and deterministic cleanup.

### Layer 4 — Recursive and compositional designs

Finally, we use `yield from` for expression trees, path-aware traversal, parsing,
pipelines, and worker supervision.

In [1]:
from __future__ import annotations

from collections.abc import Generator, Iterable, Iterator, Mapping
from dataclasses import dataclass
from numbers import Real
from typing import Any, Callable
import inspect
import operator

## A compact protocol reference

Suppose a delegator contains:

```python
result = yield from child
```

While `child` is active:

- values yielded by `child` are yielded by the delegator;
- `.send(value)` on the delegator is forwarded to `child`;
- `.throw(error)` on the delegator is forwarded to `child`;
- `.close()` on the delegator closes `child`;
- when `child` executes `return value`, the expression `yield from child`
  evaluates to `value`.

This means delegation is much richer than:

```python
for value in child:
    yield value
```

A normal loop forwards only yielded values. It does not automatically implement
the full bidirectional protocol.

# Problem 1 — Replace manual forwarding with `yield from`

We start with the simplest possible child generator: a countdown.

The child yields every integer from `start` down to `1`, then returns the string
`"finished"`.

Our first goal is to delegate to it manually. Our second goal is to replace that
manual loop with `yield from`.

## Step 1 — Write the child generator

Before reading the code, answer:

- What values will be yielded?
- What value will be stored inside `StopIteration.value`?
- Will a `for` loop reveal that returned value?

In [2]:
def countdown(start: int) -> Generator[int, None, str]:
    if start < 0:
        raise ValueError("start must be non-negative")

    for number in range(start, 0, -1):
        yield number

    return "finished"

A `for` loop sees only yielded values. It catches the final `StopIteration`
internally and discards its `.value`.

Let us confirm that.

In [3]:
assert list(countdown(4)) == [4, 3, 2, 1]
print("A list captures yielded values, not the generator's return value.")

A list captures yielded values, not the generator's return value.


## Step 2 — Capture the return value manually

To see the return value, we must call `next()` ourselves and catch
`StopIteration`.

In [4]:
def consume_with_return(
    gen: Generator[Any, Any, Any]
) -> tuple[list[Any], Any]:
    values: list[Any] = []

    while True:
        try:
            values.append(next(gen))
        except StopIteration as exc:
            return values, exc.value

In [5]:
values, final_value = consume_with_return(countdown(3))

assert values == [3, 2, 1]
assert final_value == "finished"

print(values, final_value)

[3, 2, 1] finished


## Step 3 — Delegate manually

A basic delegator can loop over the child and re-yield each value.

However, notice the limitation: the child's final return value is lost.

In [6]:
def manual_countdown_delegator(start: int) -> Generator[int, None, None]:
    for value in countdown(start):
        yield value

In [7]:
values, final_value = consume_with_return(manual_countdown_delegator(3))

assert values == [3, 2, 1]
assert final_value is None

print(values, final_value)

[3, 2, 1] None


## Step 4 — Delegate with `yield from`

Now the child return value becomes the value of the `yield from` expression.
The delegator can return it again, transform it, store it, or combine it with
other results.

In [8]:
def countdown_delegator(start: int) -> Generator[int, None, str]:
    child_result = yield from countdown(start)
    return f"child said: {child_result}"

In [9]:
values, final_value = consume_with_return(countdown_delegator(3))

assert values == [3, 2, 1]
assert final_value == "child said: finished"

print("Problem 1 tests passed.")

Problem 1 tests passed.


## What this problem established

`yield from` does two things here:

1. it forwards every yielded value;
2. it captures the child's return value.

Later problems will add `.send()`, `.throw()`, and `.close()` to the same channel.

# Problem 2 — Build a delegated calculator session

We now move from one-way iteration to bidirectional communication.

A calculator session will:

- receive commands through `.send()`;
- yield the current result after each command;
- return a session summary when it receives `"end"`.

The outer generator will run multiple calculator sessions and collect all summaries.

## Step 1 — Define the command format

Each arithmetic command is a two-item tuple:

```python
("add", 5)
("multiply", 3)
("subtract", 2)
("divide", 4)
```

The special command `"end"` ends the current session.

We will keep the protocol deliberately explicit so the caller knows exactly what
may be sent and what will be yielded.

In [10]:
OPERATIONS: dict[str, Callable[[float, float], float]] = {
    "add": operator.add,
    "subtract": operator.sub,
    "multiply": operator.mul,
    "divide": operator.truediv,
}

## Step 2 — Implement one calculator session

The first `yield` publishes the initial value and primes the coroutine.

After that:

- a normal command updates the current value;
- `"end"` returns a summary dictionary;
- invalid operations raise a clear exception.

In [11]:
def calculator_session(
    initial: float = 0.0,
) -> Generator[float, tuple[str, Real] | str, dict[str, Any]]:
    current = float(initial)
    command_count = 0

    while True:
        command = yield current

        if command == "end":
            return {
                "initial": float(initial),
                "final": current,
                "commands": command_count,
            }

        if not (
            isinstance(command, tuple)
            and len(command) == 2
            and isinstance(command[0], str)
            and isinstance(command[1], Real)
        ):
            raise TypeError("Expected ('operation', number) or 'end'.")

        operation_name, operand = command

        if operation_name not in OPERATIONS:
            raise ValueError(f"Unknown operation: {operation_name}")

        if operation_name == "divide" and operand == 0:
            raise ZeroDivisionError("Cannot divide by zero.")

        current = OPERATIONS[operation_name](current, float(operand))
        command_count += 1

## Step 3 — Interact with one session directly

Predict the returned value from every `.send()` call before running the cell.

In [12]:
session = calculator_session(initial=10)

assert next(session) == 10.0
assert session.send(("add", 5)) == 15.0
assert session.send(("multiply", 2)) == 30.0
assert session.send(("subtract", 6)) == 24.0

The final `"end"` command does not yield another value. It terminates the generator,
so the summary appears inside `StopIteration.value`.

In [13]:
try:
    session.send("end")
except StopIteration as exc:
    summary = exc.value
else:
    raise AssertionError("The session should have returned.")

assert summary == {
    "initial": 10.0,
    "final": 24.0,
    "commands": 3,
}

print(summary)

{'initial': 10.0, 'final': 24.0, 'commands': 3}


## Step 4 — Create a multi-session manager

The manager delegates to one calculator session at a time.

After each session returns, the manager yields the summary. The value sent back
after that summary determines what happens next:

- `"new"` starts another session;
- `"stop"` returns all summaries.

In [14]:
def calculator_manager() -> Generator[
    float | dict[str, Any],
    tuple[str, Real] | str,
    list[dict[str, Any]],
]:
    summaries: list[dict[str, Any]] = []

    while True:
        summary = yield from calculator_session()
        summaries.append(summary)

        decision = yield summary

        if decision == "stop":
            return summaries

        if decision != "new":
            raise ValueError("Expected 'new' or 'stop'.")

## Step 5 — Trace the delegation carefully

The first `next(manager)` enters the child session and exposes its initial result.

Commands sent to the manager are forwarded to the child until the child returns.

In [15]:
manager = calculator_manager()

assert next(manager) == 0.0
assert manager.send(("add", 8)) == 8.0
assert manager.send(("multiply", 5)) == 40.0

first_summary = manager.send("end")
assert first_summary == {
    "initial": 0.0,
    "final": 40.0,
    "commands": 2,
}

When we send `"new"`, the manager resumes after yielding the summary, loops, creates
a fresh child session, and immediately delegates to it.

In [16]:
assert manager.send("new") == 0.0
assert manager.send(("subtract", 3)) == -3.0

second_summary = manager.send("end")
assert second_summary == {
    "initial": 0.0,
    "final": -3.0,
    "commands": 1,
}

In [17]:
try:
    manager.send("stop")
except StopIteration as exc:
    all_summaries = exc.value
else:
    raise AssertionError("The manager should have returned.")

assert all_summaries == [first_summary, second_summary]
print("Problem 2 tests passed.")

Problem 2 tests passed.


## Design lesson

The manager does not implement arithmetic. It owns session sequencing.

The child session does not manage multiple sessions. It owns one arithmetic state.

`yield from` lets each generator keep one responsibility while preserving a single
interface for the caller.

# Problem 3 — Inject recoverable errors with `.throw()`

A coroutine can receive more than values. The caller may inject an exception with:

```python
generator.throw(SomeException)
```

When a delegator is currently suspended inside `yield from`, that exception is
forwarded to the active child generator.

We will build a text buffer that supports an injected `Undo` signal.

## Step 1 — Define a control exception

Using a dedicated exception class makes the protocol clear and avoids confusing
ordinary application errors with control signals.

In [18]:
class Undo(Exception):
    """Remove the most recently accepted text fragment."""

## Step 2 — Implement the child buffer

Protocol:

- send a string to append it;
- `.throw(Undo)` removes the most recent item;
- send `"commit"` to return the concatenated text;
- each successful operation yields a snapshot tuple.

In [19]:
def text_buffer() -> Generator[
    tuple[str, ...],
    str,
    str,
]:
    parts: list[str] = []

    while True:
        try:
            command = yield tuple(parts)
        except Undo:
            if parts:
                parts.pop()
            continue

        if command == "commit":
            return "".join(parts)

        if not isinstance(command, str):
            raise TypeError("Only strings may be appended.")

        parts.append(command)

## Step 3 — Add a transparent delegator

The delegator does not catch `Undo`. Therefore the exception travels through it
to the child.

In [20]:
def document_editor() -> Generator[tuple[str, ...], str, str]:
    final_document = yield from text_buffer()
    return final_document

## Step 4 — Exercise the protocol

Observe that `.throw(Undo)` itself returns the next value yielded after the child
handles the exception.

In [21]:
editor = document_editor()

assert next(editor) == ()
assert editor.send("Hello") == ("Hello",)
assert editor.send(", ") == ("Hello", ", ")
assert editor.send("wrld") == ("Hello", ", ", "wrld")

assert editor.throw(Undo) == ("Hello", ", ")
assert editor.send("world") == ("Hello", ", ", "world")
assert editor.send("!") == ("Hello", ", ", "world", "!")

In [22]:
try:
    editor.send("commit")
except StopIteration as exc:
    document = exc.value
else:
    raise AssertionError("The editor should have returned.")

assert document == "Hello, world!"
print("Problem 3 tests passed.")

Problem 3 tests passed.


## Error-handling rule

Catch only exceptions that are part of the documented protocol.

For example, `Undo` is recoverable. A programming error such as `TypeError`
should still propagate and usually terminate the generator.

# Problem 4 — Prove that `.close()` reaches the child

A generator may own resources:

- a file handle;
- a network connection;
- a lock;
- an open transaction;
- a downstream coroutine.

Cleanup belongs in `finally`.

We will simulate resource acquisition and verify the exact cleanup order.

## Step 1 — Write a child with a cleanup block

The child records lifecycle events in a list so the behavior is testable.

In [23]:
def resource_child(log: list[str]) -> Generator[str, str, None]:
    log.append("child opened")

    try:
        message = yield "child ready"

        while True:
            log.append(f"child received: {message}")
            message = yield "child ready"
    finally:
        log.append("child closed")

## Step 2 — Wrap it in two delegators

Each delegator also owns a cleanup action.

When the outermost generator closes, cleanup should occur from the inside out.

In [24]:
def middle_resource(log: list[str]) -> Generator[str, str, None]:
    log.append("middle opened")

    try:
        yield from resource_child(log)
    finally:
        log.append("middle closed")


def outer_resource(log: list[str]) -> Generator[str, str, None]:
    log.append("outer opened")

    try:
        yield from middle_resource(log)
    finally:
        log.append("outer closed")

## Step 3 — Run and close only the outermost generator

In [25]:
lifecycle: list[str] = []
resource = outer_resource(lifecycle)

assert next(resource) == "child ready"
assert resource.send("alpha") == "child ready"
assert resource.send("beta") == "child ready"

resource.close()

In [26]:
assert lifecycle == [
    "outer opened",
    "middle opened",
    "child opened",
    "child received: alpha",
    "child received: beta",
    "child closed",
    "middle closed",
    "outer closed",
]

assert inspect.getgeneratorstate(resource) == "GEN_CLOSED"

print("Problem 4 tests passed.")

Problem 4 tests passed.


## Why the order matters

The child owns the deepest resource, so it closes first.

The middle layer can safely clean up after its child is closed.

The outer layer closes last.

This inside-out structure mirrors nested context managers.

# Problem 5 — Evaluate an expression tree lazily

Recursive generators are one of the clearest uses of `yield from`.

We will evaluate a mathematical expression tree while yielding a human-readable
trace of the evaluation.

The generator will:

- yield trace messages;
- recursively delegate to child expressions;
- return the numeric value of each subtree.

## Step 1 — Define expression node types

In [27]:
@dataclass(frozen=True)
class Number:
    value: float


@dataclass(frozen=True)
class BinaryExpression:
    operator_name: str
    left: "Expression"
    right: "Expression"


Expression = Number | BinaryExpression

## Step 2 — Map operator names to functions

We keep the tree serializable and readable by storing operator names rather than
function objects.

In [28]:
BINARY_OPERATORS: dict[str, Callable[[float, float], float]] = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
}

## Step 3 — Write the recursive evaluator

Important detail:

```python
left_value = yield from evaluate(expr.left)
```

does not mean `left_value` is a yielded value. It is the return value of the child
generator after all its trace messages have been yielded.

In [29]:
def evaluate(
    expression: Expression,
) -> Generator[str, None, float]:
    if isinstance(expression, Number):
        yield f"number {expression.value}"
        return float(expression.value)

    if expression.operator_name not in BINARY_OPERATORS:
        raise ValueError(
            f"Unsupported operator: {expression.operator_name}"
        )

    yield f"enter {expression.operator_name}"

    left_value = yield from evaluate(expression.left)
    right_value = yield from evaluate(expression.right)

    if expression.operator_name == "/" and right_value == 0:
        raise ZeroDivisionError("Expression divides by zero.")

    result = BINARY_OPERATORS[expression.operator_name](
        left_value,
        right_value,
    )

    yield (
        f"apply {expression.operator_name}: "
        f"{left_value} {expression.operator_name} {right_value} = {result}"
    )

    return result

## Step 4 — Build an expression

This tree represents:

```text
(8 + 4) * (10 - 7)
```

In [30]:
expression = BinaryExpression(
    "*",
    BinaryExpression("+", Number(8), Number(4)),
    BinaryExpression("-", Number(10), Number(7)),
)

## Step 5 — Consume both outputs

The trace is the yielded stream.

The final numeric answer is the generator return value.

In [31]:
trace, result = consume_with_return(evaluate(expression))

assert trace == [
    "enter *",
    "enter +",
    "number 8",
    "number 4",
    "apply +: 8.0 + 4.0 = 12.0",
    "enter -",
    "number 10",
    "number 7",
    "apply -: 10.0 - 7.0 = 3.0",
    "apply *: 12.0 * 3.0 = 36.0",
]

assert result == 36.0

print("\n".join(trace))
print("result:", result)

enter *
enter +
number 8
number 4
apply +: 8.0 + 4.0 = 12.0
enter -
number 10
number 7
apply -: 10.0 - 7.0 = 3.0
apply *: 12.0 * 3.0 = 36.0
result: 36.0


## Why this design is useful

A normal recursive function returns one final answer.

This recursive generator returns the answer **and** streams progress information
without constructing a complete trace list internally.

# Problem 6 — Walk JSON-like data with paths

Flattening values alone is sometimes insufficient. In configuration data, logs,
and JSON documents, we often need the path to every leaf.

We will write a recursive generator that yields:

```python
(path_tuple, leaf_value)
```

Mappings contribute keys to the path. Sequences contribute numeric indices.
Strings and bytes remain atomic.

## Step 1 — Decide what counts as a container

For this exercise:

- mappings are traversed by key/value pairs;
- lists and tuples are traversed by index;
- strings, bytes, and bytearrays are leaves;
- all other values are leaves.

In [32]:
ATOMIC_TYPES = (str, bytes, bytearray)
SEQUENCE_TYPES = (list, tuple)

## Step 2 — Implement the recursive walker

In [33]:
def walk_leaves(
    value: Any,
    path: tuple[Any, ...] = (),
) -> Iterator[tuple[tuple[Any, ...], Any]]:
    if isinstance(value, ATOMIC_TYPES):
        yield path, value
        return

    if isinstance(value, Mapping):
        for key, child in value.items():
            yield from walk_leaves(child, path + (key,))
        return

    if isinstance(value, SEQUENCE_TYPES):
        for index, child in enumerate(value):
            yield from walk_leaves(child, path + (index,))
        return

    yield path, value

## Step 3 — Test nested data

In [34]:
configuration = {
    "service": {
        "name": "payments",
        "ports": [8000, 8001],
    },
    "features": (
        {"name": "audit", "enabled": True},
        {"name": "cache", "enabled": False},
    ),
}

In [35]:
leaves = list(walk_leaves(configuration))

assert leaves == [
    (("service", "name"), "payments"),
    (("service", "ports", 0), 8000),
    (("service", "ports", 1), 8001),
    (("features", 0, "name"), "audit"),
    (("features", 0, "enabled"), True),
    (("features", 1, "name"), "cache"),
    (("features", 1, "enabled"), False),
]

for path, value in leaves:
    print(path, "->", value)

('service', 'name') -> payments
('service', 'ports', 0) -> 8000
('service', 'ports', 1) -> 8001
('features', 0, 'name') -> audit
('features', 0, 'enabled') -> True
('features', 1, 'name') -> cache
('features', 1, 'enabled') -> False


## Step 4 — Add a filtering delegator

The walker should remain focused on traversal.

A second generator can delegate to it and yield only leaves matching a predicate.

In [36]:
def select_leaves(
    value: Any,
    predicate: Callable[[tuple[Any, ...], Any], bool],
) -> Iterator[tuple[tuple[Any, ...], Any]]:
    for path, leaf in walk_leaves(value):
        if predicate(path, leaf):
            yield path, leaf

In [37]:
enabled_flags = list(
    select_leaves(
        configuration,
        lambda path, leaf: path[-1:] == ("enabled",) and leaf is True,
    )
)

assert enabled_flags == [
    (("features", 0, "enabled"), True),
]

print("Problem 6 tests passed.")

Problem 6 tests passed.


## Composition lesson

The recursive walker handles structure.

The filtering generator handles selection.

Keeping these responsibilities separate makes both components easier to test and
reuse.

# Problem 7 — Add depth control and cycle detection

Real object graphs may contain:

- very deep nesting;
- shared references;
- direct cycles;
- indirect cycles.

A naïve recursive generator can recurse forever on cyclic data.

We will extend a nested-sequence flattener with:

- atomic string handling;
- maximum depth;
- active-path cycle detection.

## Step 1 — Understand active-path detection

A global set of every previously seen object would be too strict.

The same list may legally appear in two different branches:

```python
shared = [1, 2]
data = [shared, shared]
```

That is sharing, not a cycle.

A cycle exists only when the same container appears again on the **currently active
recursive path**.

## Step 2 — Implement the guarded flattener

In [38]:
def flatten_guarded(
    value: Any,
    *,
    max_depth: int | None = None,
    _depth: int = 0,
    _active: set[int] | None = None,
) -> Iterator[Any]:
    if max_depth is not None and max_depth < 0:
        raise ValueError("max_depth must be non-negative or None.")

    if isinstance(value, ATOMIC_TYPES):
        yield value
        return

    if max_depth is not None and _depth >= max_depth:
        yield value
        return

    try:
        iterator = iter(value)
    except TypeError:
        yield value
        return

    if _active is None:
        _active = set()

    object_id = id(value)

    if object_id in _active:
        raise ValueError("Cycle detected.")

    _active.add(object_id)

    try:
        for child in iterator:
            yield from flatten_guarded(
                child,
                max_depth=max_depth,
                _depth=_depth + 1,
                _active=_active,
            )
    finally:
        _active.remove(object_id)

## Step 3 — Verify normal nesting

In [39]:
assert list(
    flatten_guarded([1, (2, [3, "abc"]), 4])
) == [1, 2, 3, "abc", 4]

## Step 4 — Verify shared references are allowed

In [40]:
shared = [1, 2]
not_a_cycle = [shared, shared]

assert list(flatten_guarded(not_a_cycle)) == [1, 2, 1, 2]

## Step 5 — Verify cycle detection

In [41]:
cycle: list[Any] = ["start"]
cycle.append(cycle)

try:
    list(flatten_guarded(cycle))
except ValueError as exc:
    assert str(exc) == "Cycle detected."
else:
    raise AssertionError("The cycle should have been detected.")

## Step 6 — Verify depth limiting

At the depth boundary, the nested value is yielded as a leaf instead of traversed.

In [42]:
depth_limited = list(
    flatten_guarded([1, [2, [3, [4]]]], max_depth=2)
)

assert depth_limited == [1, 2, [3, [4]]]

print("Problem 7 tests passed.")

Problem 7 tests passed.


# Problem 8 — Parse nested parentheses with recursive delegation

A parser is a natural fit for recursive generators.

We will parse a token stream containing integers and parentheses.

Example input:

```python
[1, "(", 2, "(", 3, 4, ")", 5, ")", 6]
```

Expected structure:

```python
[1, [2, [3, 4], 5], 6]
```

The parser will yield trace messages and return the parsed list.

## Step 1 — Create a token cursor

The cursor gives us controlled access to the token sequence and a consistent place
to report unexpected end-of-input.

In [43]:
@dataclass
class TokenCursor:
    tokens: list[Any]
    position: int = 0

    def has_next(self) -> bool:
        return self.position < len(self.tokens)

    def next(self) -> Any:
        if not self.has_next():
            raise ValueError("Unexpected end of token stream.")

        token = self.tokens[self.position]
        self.position += 1
        return token

## Step 2 — Write a recursive group parser

The parser stops when it reaches:

- `")"` for a nested group;
- end-of-input for the top-level group.

When it sees `"("`, it delegates to a new recursive group parser.

In [44]:
def parse_group(
    cursor: TokenCursor,
    *,
    nested: bool,
) -> Generator[str, None, list[Any]]:
    result: list[Any] = []

    while cursor.has_next():
        token = cursor.next()

        if token == "(":
            yield "enter group"
            child = yield from parse_group(cursor, nested=True)
            result.append(child)
            yield "leave group"
            continue

        if token == ")":
            if not nested:
                raise ValueError("Unexpected closing parenthesis.")
            return result

        result.append(token)
        yield f"accept {token!r}"

    if nested:
        raise ValueError("Missing closing parenthesis.")

    return result

## Step 3 — Add a top-level parser

In [45]:
def parse_parenthesized(
    tokens: Iterable[Any],
) -> Generator[str, None, list[Any]]:
    cursor = TokenCursor(list(tokens))
    parsed = yield from parse_group(cursor, nested=False)
    return parsed

## Step 4 — Parse and inspect the trace

In [46]:
tokens = [1, "(", 2, "(", 3, 4, ")", 5, ")", 6]

parse_trace, parsed = consume_with_return(
    parse_parenthesized(tokens)
)

assert parsed == [1, [2, [3, 4], 5], 6]

assert parse_trace == [
    "accept 1",
    "enter group",
    "accept 2",
    "enter group",
    "accept 3",
    "accept 4",
    "leave group",
    "accept 5",
    "leave group",
    "accept 6",
]

print("\n".join(parse_trace))
print(parsed)

accept 1
enter group
accept 2
enter group
accept 3
accept 4
leave group
accept 5
leave group
accept 6
[1, [2, [3, 4], 5], 6]


## Step 5 — Test malformed input

In [47]:
try:
    consume_with_return(
        parse_parenthesized([1, "(", 2])
    )
except ValueError as exc:
    assert str(exc) == "Missing closing parenthesis."
else:
    raise AssertionError("Malformed input should fail.")

print("Problem 8 tests passed.")

Problem 8 tests passed.


## Parser lesson

Each recursive parser returns one completed subtree.

The parent receives that subtree from `yield from`, appends it, and continues parsing
its own level.

# Problem 9 — Build a configurable coroutine pipeline

We will create a push-style pipeline with three stages:

1. scale numbers;
2. keep numbers above a threshold;
3. collect accepted values.

The outermost coroutine will delegate to the first stage. Values sent to the outer
generator travel through all stages.

Unlike a normal iterator pipeline, data moves inward through `.send()`.

## Step 1 — Write the sink

The sink is the final consumer. It appends every received value to an output list.

In [48]:
def number_sink(
    output: list[float],
    lifecycle: list[str],
) -> Generator[None, float, None]:
    try:
        while True:
            value = yield
            output.append(value)
    finally:
        lifecycle.append("sink closed")

## Step 2 — Write the filtering stage

A stage owns its downstream coroutine:

- it primes it once;
- it forwards selected values;
- it closes it in `finally`.

In [49]:
def threshold_stage(
    threshold: float,
    downstream: Generator[None, float, None],
    lifecycle: list[str],
) -> Generator[None, float, None]:
    next(downstream)

    try:
        while True:
            value = yield

            if value >= threshold:
                downstream.send(value)
    finally:
        downstream.close()
        lifecycle.append("threshold closed")

## Step 3 — Write the scaling stage

In [50]:
def scale_stage(
    factor: float,
    downstream: Generator[None, float, None],
    lifecycle: list[str],
) -> Generator[None, Real, None]:
    next(downstream)

    try:
        while True:
            value = yield

            if not isinstance(value, Real):
                raise TypeError("Pipeline accepts only real numbers.")

            downstream.send(float(value) * factor)
    finally:
        downstream.close()
        lifecycle.append("scale closed")

## Step 4 — Build the outer application

The outer generator creates and connects the stages, then delegates to the first
stage.

In [51]:
def number_pipeline(
    output: list[float],
    lifecycle: list[str],
    *,
    factor: float,
    threshold: float,
) -> Generator[None, Real, None]:
    sink = number_sink(output, lifecycle)
    filtered = threshold_stage(threshold, sink, lifecycle)

    yield from scale_stage(
        factor,
        filtered,
        lifecycle,
    )

## Step 5 — Send data through the complete pipeline

With `factor=2` and `threshold=10`:

- `2` becomes `4` and is rejected;
- `5` becomes `10` and is accepted;
- `8` becomes `16` and is accepted.

In [52]:
pipeline_output: list[float] = []
pipeline_lifecycle: list[str] = []

pipeline = number_pipeline(
    pipeline_output,
    pipeline_lifecycle,
    factor=2,
    threshold=10,
)

next(pipeline)

pipeline.send(2)
pipeline.send(5)
pipeline.send(8)

pipeline.close()

assert pipeline_output == [10.0, 16.0]
assert pipeline_lifecycle == [
    "sink closed",
    "threshold closed",
    "scale closed",
]

print("Problem 9 tests passed.")

Problem 9 tests passed.


## Ownership lesson

Every stage closes only the downstream stage it owns.

The caller closes only the outermost pipeline.

This avoids requiring callers to know the internal topology.

# Problem 10 — Create a batching subgenerator that returns statistics

A batching component will receive numbers until one of two events occurs:

- the batch reaches a fixed size;
- the caller sends `FLUSH`.

The subgenerator returns a batch report. A supervisor runs repeated batches and
yields each report.

## Step 1 — Define a flush sentinel and report type

In [53]:
FLUSH = object()


@dataclass(frozen=True)
class BatchReport:
    values: tuple[float, ...]
    count: int
    total: float
    average: float | None

## Step 2 — Implement one batch

The batch generator yields `None` while waiting for values.

When complete, it returns a report rather than yielding it.

In [54]:
def collect_fixed_batch(
    size: int,
) -> Generator[None, Real | object, BatchReport]:
    if size <= 0:
        raise ValueError("size must be positive.")

    values: list[float] = []

    while len(values) < size:
        item = yield

        if item is FLUSH:
            break

        if not isinstance(item, Real):
            raise TypeError("Batch values must be real numbers.")

        values.append(float(item))

    total = sum(values)

    return BatchReport(
        values=tuple(values),
        count=len(values),
        total=total,
        average=(total / len(values)) if values else None,
    )

## Step 3 — Implement the supervisor

After yielding a report, the supervisor expects:

- `"continue"` to start another batch;
- `"stop"` to return all reports.

In [55]:
def batching_supervisor(
    size: int,
) -> Generator[
    None | BatchReport,
    Real | object | str,
    list[BatchReport],
]:
    reports: list[BatchReport] = []

    while True:
        report = yield from collect_fixed_batch(size)
        reports.append(report)

        command = yield report

        if command == "stop":
            return reports

        if command != "continue":
            raise ValueError("Expected 'continue' or 'stop'.")

## Step 4 — Complete a full batch

In [56]:
batcher = batching_supervisor(size=3)

assert next(batcher) is None
assert batcher.send(2) is None
assert batcher.send(4) is None

report_1 = batcher.send(6)

assert report_1 == BatchReport(
    values=(2.0, 4.0, 6.0),
    count=3,
    total=12.0,
    average=4.0,
)

## Step 5 — Flush a partial batch

In [57]:
assert batcher.send("continue") is None
assert batcher.send(10) is None

report_2 = batcher.send(FLUSH)

assert report_2 == BatchReport(
    values=(10.0,),
    count=1,
    total=10.0,
    average=10.0,
)

In [58]:
try:
    batcher.send("stop")
except StopIteration as exc:
    reports = exc.value
else:
    raise AssertionError("The supervisor should have returned.")

assert reports == [report_1, report_2]

print("Problem 10 tests passed.")

Problem 10 tests passed.


## Why return the report?

The report is the result of one subgenerator lifecycle.

Returning it lets the supervisor decide what to do next:

- store it;
- aggregate it;
- yield it;
- retry;
- stop.

# Problem 11 — Inspect a live delegation chain

Debugging nested delegation can be difficult because the caller interacts only with
the outer generator.

On CPython, a generator's `gi_yieldfrom` attribute points to its currently active
delegated iterator or generator.

We will inspect a three-level chain.

## Step 1 — Create the chain

In [59]:
def deepest() -> Generator[str, str, None]:
    incoming = yield "deepest ready"

    while True:
        incoming = yield f"deepest received {incoming}"


def middle() -> Generator[str, str, None]:
    yield from deepest()


def outermost() -> Generator[str, str, None]:
    yield from middle()

## Step 2 — Write an inspection helper

In [60]:
def active_generator_chain(
    root: Generator[Any, Any, Any],
) -> list[tuple[str, str]]:
    chain: list[tuple[str, str]] = []
    current: Any = root

    while inspect.isgenerator(current):
        chain.append(
            (
                current.gi_code.co_name,
                inspect.getgeneratorstate(current),
            )
        )
        current = current.gi_yieldfrom

    return chain

## Step 3 — Inspect before and after priming

In [61]:
root = outermost()

assert active_generator_chain(root) == [
    ("outermost", "GEN_CREATED"),
]

assert next(root) == "deepest ready"

assert active_generator_chain(root) == [
    ("outermost", "GEN_SUSPENDED"),
    ("middle", "GEN_SUSPENDED"),
    ("deepest", "GEN_SUSPENDED"),
]

## Step 4 — Verify that sending to the root reaches the leaf

In [62]:
assert root.send("hello") == "deepest received hello"

root.close()

assert inspect.getgeneratorstate(root) == "GEN_CLOSED"

print("Problem 11 tests passed.")

Problem 11 tests passed.


## Portability note

Generator inspection is excellent for debugging and teaching.

Application behavior should not normally depend on CPython-specific generator
internals unless that dependency is deliberate and documented.

# Problem 12 — Build a fault-tolerant delegated worker

A worker processes jobs sent through `.send()`.

Some failures are recoverable. We will inject `SkipJob` with `.throw()` to tell the
worker to discard the current job and continue.

A supervisor delegates to the worker and returns the final processing report.

## Step 1 — Define the control exception and report

In [63]:
class SkipJob(Exception):
    """Discard the current pending job."""


@dataclass(frozen=True)
class WorkerReport:
    completed: tuple[str, ...]
    skipped: int

## Step 2 — Implement the worker protocol

The worker yields a status string.

Protocol:

- send a job string;
- the worker yields `"processed: <job>"`;
- `.throw(SkipJob)` increments the skipped counter;
- send `"shutdown"` to return the report.

In [64]:
def worker() -> Generator[
    str,
    str,
    WorkerReport,
]:
    completed: list[str] = []
    skipped = 0

    while True:
        try:
            job = yield "waiting"
        except SkipJob:
            skipped += 1
            continue

        if job == "shutdown":
            return WorkerReport(
                completed=tuple(completed),
                skipped=skipped,
            )

        if not isinstance(job, str):
            raise TypeError("Jobs must be strings.")

        completed.append(job)
        yield f"processed: {job}"

The worker yields twice for each normal job:

1. `"waiting"` before accepting the job;
2. `"processed: ..."` after completing it.

After the processed status, the next `next()` or `.send(None)` moves it back to
the waiting state.

## Step 3 — Add the supervisor

In [65]:
def worker_supervisor() -> Generator[
    str,
    str,
    WorkerReport,
]:
    report = yield from worker()
    return report

## Step 4 — Exercise normal processing and skipping

In [66]:
supervisor = worker_supervisor()

assert next(supervisor) == "waiting"

assert supervisor.send("job-a") == "processed: job-a"
assert next(supervisor) == "waiting"

assert supervisor.throw(SkipJob) == "waiting"

assert supervisor.send("job-b") == "processed: job-b"
assert next(supervisor) == "waiting"

In [67]:
try:
    supervisor.send("shutdown")
except StopIteration as exc:
    worker_report = exc.value
else:
    raise AssertionError("The supervisor should have returned.")

assert worker_report == WorkerReport(
    completed=("job-a", "job-b"),
    skipped=1,
)

print("Problem 12 tests passed.")

Problem 12 tests passed.


## Protocol design lesson

Coroutines are easiest to use when every suspension point has a documented meaning.

In this worker:

- `"waiting"` means the next sent value is interpreted as a job;
- `"processed: ..."` means the previous job finished;
- the caller must resume once more before sending the next job.

Ambiguous suspension points are a common source of coroutine bugs.

# Capstone — A delegated command router with sub-sessions

We will combine the notebook's main ideas.

The router accepts commands for two independent sub-sessions:

- a numeric accumulator;
- a text collector.

Each session:

- receives values through `.send()`;
- yields live state;
- returns a final summary.

The router delegates to one active session at a time and stores all summaries.

## Step 1 — Numeric session

In [68]:
def numeric_session() -> Generator[
    float,
    Real | str,
    dict[str, Any],
]:
    total = 0.0
    count = 0

    while True:
        command = yield total

        if command == "done":
            return {
                "kind": "numeric",
                "total": total,
                "count": count,
            }

        if not isinstance(command, Real):
            raise TypeError("Numeric session expects real numbers.")

        total += float(command)
        count += 1

## Step 2 — Text session

In [69]:
def string_session() -> Generator[
    tuple[str, ...],
    str,
    dict[str, Any],
]:
    values: list[str] = []

    while True:
        command = yield tuple(values)

        if command == "done":
            return {
                "kind": "text",
                "values": tuple(values),
                "joined": " ".join(values),
            }

        if not isinstance(command, str):
            raise TypeError("Text session expects strings.")

        values.append(command)

## Step 3 — Router protocol

The router initially yields `"choose"`.

The caller then sends:

- `"numeric"` to start a numeric session;
- `"text"` to start a text session;
- `"stop"` to return all previous summaries.

After a sub-session returns, the router yields its summary and then returns to the
`"choose"` state.

In [70]:
def command_router() -> Generator[
    str | float | tuple[str, ...] | dict[str, Any],
    Any,
    list[dict[str, Any]],
]:
    summaries: list[dict[str, Any]] = []

    while True:
        choice = yield "choose"

        if choice == "stop":
            return summaries

        if choice == "numeric":
            summary = yield from numeric_session()
        elif choice == "text":
            summary = yield from string_session()
        else:
            raise ValueError("Expected 'numeric', 'text', or 'stop'.")

        summaries.append(summary)
        yield summary

## Step 4 — Run a numeric sub-session

In [71]:
router = command_router()

assert next(router) == "choose"
assert router.send("numeric") == 0.0
assert router.send(5) == 5.0
assert router.send(2.5) == 7.5

numeric_summary = router.send("done")

assert numeric_summary == {
    "kind": "numeric",
    "total": 7.5,
    "count": 2,
}

After yielding the summary, the router must resume once to reach the next `"choose"`
suspension point.

In [72]:
assert next(router) == "choose"

## Step 5 — Run a text sub-session

In [73]:
assert router.send("text") == ()
assert router.send("advanced") == ("advanced",)
assert router.send("generators") == ("advanced", "generators")

text_summary = router.send("done")

assert text_summary == {
    "kind": "text",
    "values": ("advanced", "generators"),
    "joined": "advanced generators",
}

## Step 6 — Stop the router and capture all summaries

In [74]:
assert next(router) == "choose"

try:
    router.send("stop")
except StopIteration as exc:
    router_report = exc.value
else:
    raise AssertionError("The router should have returned.")

assert router_report == [
    numeric_summary,
    text_summary,
]

print("Capstone tests passed.")

Capstone tests passed.


# Common mistakes and corrections

## Mistake 1 — Sending before priming

Incorrect:

```python
g = coroutine()
g.send(10)
```

A newly created generator cannot receive a non-`None` value.

Correct:

```python
next(g)
g.send(10)
```

---

## Mistake 2 — Assuming a `for` loop preserves return values

A `for` loop discards `StopIteration.value`.

Use `yield from` inside another generator or catch `StopIteration` manually.

---

## Mistake 3 — Putting cleanup after an infinite loop

Code after an infinite coroutine loop may never execute.

Use:

```python
try:
    ...
finally:
    cleanup()
```

---

## Mistake 4 — Treating strings as generic nested iterables

A recursive flattener may repeatedly descend into one-character strings.

Handle strings and bytes as atomic values unless character-level traversal is
explicitly required.

---

## Mistake 5 — Hiding an unclear protocol

Every `yield` is a suspension point. Document what the next `.send()` means at each
suspension point.

---

## Mistake 6 — Catching every exception

Do not use broad exception handling to keep a coroutine alive at all costs.

Catch only errors the coroutine can recover from correctly.

# Final checklist

Before using `yield from` in production code, ask:

1. What values does the child yield?
2. What values may the caller send?
3. What does the child return?
4. Which exceptions are part of the protocol?
5. Which exceptions should terminate the generator?
6. Who primes each coroutine?
7. Who owns and closes each downstream generator?
8. Are strings, mappings, and cycles handled intentionally?
9. Is recursion depth bounded?
10. Would a normal function or iterator class be clearer?

`yield from` is most valuable when it creates a clean boundary between components
while preserving the full generator protocol.

# End-of-notebook summary

This tutorial demonstrated that `yield from` supports:

- simple value forwarding;
- subgenerator return-value capture;
- bidirectional `.send()` communication;
- exception injection with `.throw()`;
- transitive cleanup with `.close()`;
- recursive computation with streamed trace output;
- path-aware traversal;
- recursive parsing;
- push-style coroutine pipelines;
- batch supervision;
- live delegation inspection;
- fault-tolerant worker control;
- multi-session command routing.

The central design idea is separation of responsibilities:

- child generators own local state and local rules;
- delegators own sequencing, aggregation, and lifecycle;
- callers interact with one stable outer interface.